In [3]:
!pip install chromadb

In [4]:
from __future__ import annotations
import os
import uuid
import chromadb
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader

In [5]:
# ---------------- Config ----------------
PERSIST_DIR = "./chroma_multimodal_local_dataloader" # เปลี่ยน path เพื่อไม่ให้ปนกับของเก่า
COLLECTION_NAME = "pets_local_dataloader"

DOG_IMAGES = ["dog1.png", "dog2.png", "dog3.png"]
CAT_IMAGES = ["cat1.png", "cat2.png", "cat3.png"]

In [6]:
# Step 1
all_image_paths = DOG_IMAGES + CAT_IMAGES
labels = ["dog"] * len(DOG_IMAGES) + ["cat"] * len(CAT_IMAGES)
doc_ids = [str(uuid.uuid4()) for _ in all_image_paths]

document_descriptions = [f"A photo of a {lbl}" for lbl in labels]
metadatas = [
    {
        "label": lbl,
        "filename": os.path.basename(path),
        "description": desc
    }
    for lbl, path, desc in zip(labels, all_image_paths, document_descriptions)
]

In [8]:
!pip install open-clip-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [9]:
# Step 2 - Setup ChromaDB Client

client = chromadb.PersistentClient(path=PERSIST_DIR)

# Step 3
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenCLIPEmbeddingFunction(), # บอกให้ใช้ OpenCLIP
    data_loader=ImageLoader(),                     # บอกให้ใช้ ImageLoader
    metadata={"hnsw:space": "cosine"}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [10]:
# Step 4
collection.add(
    ids=doc_ids,
    uris=all_image_paths, # <--- จุดสำคัญคือตรงนี้
    metadatas=metadatas
)
print(f"Indexed {len(doc_ids)} images into collection '{collection.name}'.")

Indexed 6 images into collection 'pets_local_dataloader'.


In [12]:
# Step 5
query_text = "dog"
res = collection.query(
  query_texts=[query_text],
  n_results=6,
  include=["metadatas", "documents", "distances"]
)


print(f"\n=== Query: '{query_text}' ===")
for rank, (meta, doc, dist) in enumerate(zip(res["metadatas"][0], res["documents"][0], res["distances"][0]), start=1):
  print(f"#{rank} -> {meta['filename']} ({meta['label']}) distance={dist:.4f}")
  print("desc:", doc)


=== Query: 'dog' ===
#1 -> dog1.png (dog) distance=0.7248
desc: None
#2 -> dog2.png (dog) distance=0.7345
desc: None
#3 -> dog3.png (dog) distance=0.7376
desc: None
#4 -> cat1.png (cat) distance=0.7784
desc: None
#5 -> cat2.png (cat) distance=0.8142
desc: None
#6 -> cat3.png (cat) distance=0.8169
desc: None
